In [34]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

In [35]:
df = pd.read_csv('alphatrade_phase1.csv')

print(df.shape)
df.head()

(1508, 19)


,Date,Close,High,Low,Open,Volume,SMA_20,SMA_50,EMA_20,EMA_50,Daily_Return,Volatility,RSI,MACD,Signal_Line,BB_Middle,BB_Upper,BB_Lower,Signal
0,2020-01-02,72.333855,72.394063,71.091161,71.344032,135480400,NaN,NaN,72.333855,72.333855,NaN,NaN,NaN,0.000000,0.000000,NaN,NaN,NaN,0
1,2020-01-03,71.630630,72.389250,71.406659,71.563198,146322800,NaN,NaN,72.266881,72.306277,-0.009722,NaN,NaN,-0.056098,-0.011220,NaN,NaN,NaN,0
2,2020-01-06,72.201401,72.239935,70.503539,70.754006,118387200,NaN,NaN,72.260645,72.302164,0.007968,NaN,NaN,-0.053878,-0.019751,NaN,NaN,NaN,0
3,2020-01-07,71.861855,72.466338,71.642697,72.211056,108872000,NaN,NaN,72.222665,72.284897,-0.004703,NaN,NaN,-0.078611,-0.031523,NaN,NaN,NaN,0
4,2020-01-08,73.017815,73.318854,71.565599,71.565599,132079200,NaN,NaN,72.298393,72.313639,0.016086,NaN,NaN,-0.004880,-0.026195,NaN,NaN,NaN,0


In [36]:
# Create Target Again
df['Target'] = (
    df['Close'].shift(-1)
    > df['Close']
).astype(int)

df = df.dropna()

print(df.shape)
print(df.isnull().sum())

(1459, 20)
Date            0
Close           0
High            0
Low             0
Open            0
Volume          0
SMA_20          0
SMA_50          0
EMA_20          0
EMA_50          0
Daily_Return    0
Volatility      0
RSI             0
MACD            0
Signal_Line     0
BB_Middle       0
BB_Upper        0
BB_Lower        0
Signal          0
Target          0
dtype: int64


In [37]:
features = [
    'Close',
    'Volume',
    'SMA_20',
    'SMA_50',
    'EMA_20',
    'EMA_50',
    'Daily_Return',
    'Volatility',
    'RSI',
    'MACD',
    'Signal_Line'
]

X = df[features]
y = df['Target']

In [38]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

print(np.isnan(X_scaled).sum())

0


In [39]:
'Normalize Data LSTMs require scaled inputs.'
scaler = MinMaxScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)

(1459, 11)


In [40]:
X_seq = []
y_seq = []

lookback = 10

for i in range(lookback, len(X_scaled)):
    X_seq.append(X_scaled[i-lookback:i])
    y_seq.append(y.iloc[i])

X_seq = np.array(X_seq)
y_seq = np.array(y_seq)

print(X_seq.shape)

(1449, 10, 11)


In [41]:
print(X_seq.shape)
print(y_seq.shape)

(1449, 10, 11)
(1449,)


In [42]:
'''Chronological Split

Since this is a time-series problem, we again use chronological splitting.'''
split = int(len(X_seq) * 0.8)

X_train = X_seq[:split]
X_test = X_seq[split:]

y_train = y_seq[:split]
y_test = y_seq[split:]

LSTM Model:

In [43]:
model = Sequential([
    Input(shape=(10,11)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

In [44]:
# Compile:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [45]:
history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

Epoch 1/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 13s 43ms/step - accuracy: 0.5151 - loss: 0.6949 - val_accuracy: 0.5414 - val_loss: 0.6916
Epoch 2/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.5246 - loss: 0.6929 - val_accuracy: 0.5414 - val_loss: 0.6904
Epoch 3/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.5298 - loss: 0.6917 - val_accuracy: 0.5414 - val_loss: 0.6910
Epoch 4/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.5280 - loss: 0.6915 - val_accuracy: 0.5414 - val_loss: 0.6904
Epoch 5/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.5427 - loss: 0.6912 - val_accuracy: 0.4862 - val_loss: 0.6936
Epoch 6/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - accuracy: 0.5160 - loss: 0.6924 - val_accuracy: 0.5414 - val_loss: 0.6913
Epoch 7/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 44ms/step - accuracy: 0.5324 - loss: 0.6918 - val_accuracy: 0.5414 - val_loss: 0.6902
Epoch 8/30
37/37 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - accuracy: 0.5203 - loss: 0.6918 - val_accuracy: 0.5414 - 